In [ ]:
import pandas as pd
import numpy as np
import re
import lightgbm as lgb
from sklearn.model_selection import KFold
import torch
from sentence_transformers import SentenceTransformer
import timm
from PIL import Image
from torchvision import transforms
import os
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings('ignore')

print("--- Starting V2 Model Training Pipeline: Now with Advanced Features ---")

DATASET_FOLDER = 'dataset/'
IMAGE_DOWNLOAD_FOLDER = 'images/' # We will load images from here
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
try:
    train_df = pd.read_csv(os.path.join(DATASET_FOLDER, 'train.csv'))
    test_df = pd.read_csv(os.path.join(DATASET_FOLDER, 'test.csv'))
    print(f"Data loaded. Train shape: {train_df.shape}, Test shape: {test_df.shape}")
except FileNotFoundError:
    print(f"Error: Make sure 'train.csv' and 'test.csv' are in the '{DATASET_FOLDER}' folder.")
    exit()

# Combine for easier feature engineering
all_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

print("\n--- Generating Advanced Numerical Features ---")

def extract_numerical_features(df):
    weights, counts, volumes = [], [], []
    
    # Regex patterns for common units (handles integers, decimals, and optional spaces)
    weight_pattern = r'(\d+\.?\d*)\s*(oz|ounce|lb|pound|g|gram|kg|kilogram)'
    count_pattern = r'(\d+)\s*(count|pack|ct)'
    volume_pattern = r'(\d+\.?\d*)\s*(ml|milliliter|l|liter|fl oz|fluid ounce)'

    for text in tqdm(df['catalog_content'].str.lower(), desc="Extracting Numerical Features"):
        if not isinstance(text, str):
            weights.append(0)
            counts.append(0)
            volumes.append(0)
            continue
            
        found_weights = re.findall(weight_pattern, text)
        found_counts = re.findall(count_pattern, text)
        found_volumes = re.findall(volume_pattern, text)
        
        weights.append(float(found_weights[0][0]) if found_weights else 0)
        counts.append(int(found_counts[0][0]) if found_counts else 0)
        volumes.append(float(found_volumes[0][0]) if found_volumes else 0)
        
    df['feature_weight'] = weights
    df['feature_count'] = counts
    df['feature_volume'] = volumes
    
    print("Numerical features extracted successfully.")
    return df

all_df = extract_numerical_features(all_df)

print("\n--- Generating Text Embeddings ---")
text_model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

def clean_text(text):
    if not isinstance(text, str): return ""
    return re.sub(r'\s+', ' ', text).strip()

all_df['cleaned_content'] = all_df['catalog_content'].apply(clean_text)
text_embeddings = text_model.encode(all_df['cleaned_content'].tolist(), show_progress_bar=True, batch_size=128)
print(f"Text embeddings generated. Shape: {text_embeddings.shape}")
print("\n--- Generating Image Embeddings from Local Files ---")
image_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0).to(DEVICE)
image_model.eval()
data_config = timm.data.resolve_model_data_config(image_model)
image_transforms = timm.data.create_transform(**data_config, is_training=False)

def get_image_embedding_from_local(image_link):
    """Loads a pre-downloaded image and extracts features."""
    try:
        image_filename = os.path.join(IMAGE_DOWNLOAD_FOLDER, str(image_link.split('/')[-1]))
        if not os.path.exists(image_filename):
             # If an image is missing, return a vector of zeros.
            return np.zeros(1280)
            
        image = Image.open(image_filename).convert("RGB")
        transformed_image = image_transforms(image).unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            embedding = image_model(transformed_image).cpu().numpy().squeeze()
        return embedding
    except Exception:
        # If image is corrupted or another error occurs, return zeros.
        return np.zeros(1280)

image_embeddings = np.array([get_image_embedding_from_local(link) for link in tqdm(all_df['image_link'], desc="Processing Images")])
print(f"Image embeddings generated. Shape: {image_embeddings.shape}")

print("\n--- Combining All Features ---")
new_numerical_features = all_df[['feature_weight', 'feature_count', 'feature_volume']].values

# Combine text embeddings, image embeddings, AND our new numerical features
combined_features = np.hstack([text_embeddings, image_embeddings, new_numerical_features])
print(f"Final combined features shape: {combined_features.shape}")

# Split back into training and testing sets
X = combined_features[:len(train_df)]
X_test = combined_features[len(train_df):]
y = np.log1p(train_df['price'])

print("\n--- Training Model with 5-Fold Cross-Validation ---")
lgb_params = {
    'objective': 'regression_l1', 'metric': 'mae', 'n_estimators': 2000,
    'learning_rate': 0.01, 'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'bagging_freq': 1, 'lambda_l1': 0.1, 'lambda_l2': 0.1,
    'num_leaves': 31, 'verbose': -1, 'n_jobs': -1, 'seed': 42,
}

NFOLDS = 5
folds = KFold(n_splits=NFOLDS, shuffle=True, random_state=42)
test_preds = np.zeros(X_test.shape[0])

for fold_, (trn_idx, val_idx) in enumerate(folds.split(X, y)):
    print(f"Current Fold: {fold_}")
    trn_data = lgb.Dataset(X[trn_idx], label=y.iloc[trn_idx])
    val_data = lgb.Dataset(X[val_idx], label=y.iloc[val_idx])

    clf = lgb.train(
        lgb_params, trn_data,
        valid_sets=[trn_data, val_data],
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    
    test_preds += clf.predict(X_test, num_iteration=clf.best_iteration) / folds.n_splits

print("\n--- Creating Submission File ---")
final_predictions = np.expm1(test_preds)
final_predictions[final_predictions < 0] = 0

submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': final_predictions
})

submission_df.to_csv('test_out.csv', index=False)
print("✅ Submission file 'test_out.csv' created successfully!")
print(submission_df.head())

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Starting Optimal Model Training Pipeline ---
Using device: cpu
Data loaded. Train shape: (75000, 4), Test shape: (75000, 3)

--- Generating Text Embeddings ---


Batches:   0%|                                                                                                                                                                        | 0/1172 [00:00<?, ?it/s]